# [실습 프로젝트] Naive RAG 구현 

- 각 단계별 지시사항에 따라 코드를 완성하세요. 
- 제시된 지시사항과 LangChain 문서를 참조하여 시스템을 구성합니다. 

## 0. 환경 설정 및 준비

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [7]:
import os
from pprint import pprint

## 1. 문서 준비/로드
### PDF

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 로더 초기화
pdf_loader = PyPDFLoader('.././data/membersystem_access_rules.pdf')

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')

PDF 문서 개수: 15


In [3]:
# 첫 번째 문서의 내용 출력
print(pdf_docs[0].page_content)

- 1 -
회원시스템접속등에관한지침제정 2011. 12. 28. 기준 제265호개정 2012. 4. 27. 기준 제284호개정 2013. 5. 3. 기준 제328호개정 2013. 9. 16. 기준 제350호개정 2014. 2. 28. 기준 제377호개정 2014. 3. 20. 기준 제381호개정 2015. 4. 13. 기준 제442호개정 2015. 8. 7. 기준 제461호개정 2021. 6. 14. 지침 제739호개정 2022. 5. 18. 지침 제776호개정 2022. 12. 28. 지침 제813호개정 2025. 5. 30. 지침 제932호개정 2025. 11. 12. 지침 제(현행)968호개정 2025. 12. 15. 지침 제975호제1장 총칙제1조(목적) 이 지침은 한국거래소 「유가증권시장 업무규정」 제8조의2, 「코스닥시장 업무규정」 제7조의3, 「코넥스시장 업무규정」 제9조, 「파생상품시장 업무규정」 제8조의2, 「KRX금시장 운영규정」 제11조 및 「배출권 거래시장 운영규정」제10조의2에 따라 회원과 한국거래소 간 및 회원과 위탁자 간의 시스템 연결 등에 관하여 필요한 사항을 규정함을 목적으로 한다. <개정 2013. 9. 16., 2014. 3. 20., 2021. 6. 14.,2025. 11. 12.>제2조(정의) ① 이 지침에서 “회원”이란 한국거래소(이하 “거래소”라 한다) 「회원관리규정」 제3조제1항 각 호, 「KRX금시장 운영규정」제12조제1항 각 호 또는 「배출권 거래시장 운영규정」제11조의2제3호에 따른 회원을 말한다. <개정 2014. 3. 20., 2021. 6. 14.,2025. 11. 12.>② 이 지침에서 “거래소시스템”이란 거래소가 개설한 증권시장, 파생상품시장, KRX금시장 및 배출권 거래시장에서의 거래 등을 위하여 거래소가 운영하는 시스템을 말한다. <개정 2013. 9. 16.,


In [8]:
# 첫 번째 문서의 메타데이터 출력
pprint(pdf_docs[0].metadata)

{'author': '',
 'creationdate': '',
 'creator': 'Call PDF',
 'page': 0,
 'page_label': '1',
 'producer': 'Call PDF v 2.4',
 'source': '.././data/membersystem_access_rules.pdf',
 'subject': '',
 'title': '',
 'total_pages': 15}


### text

In [12]:
# TXT 파일 로드
with open(".././data/derivatives_rules_details.txt", "r") as f:
    text_content = f.read()

from langchain_core.documents import Document

# 단일 Document 객체 생성
text_doc = Document(
    page_content=text_content, 
    metadata={"source": "파생상품시장 업무규정 시행세칙_전문"}
)

In [13]:
# PDF와 TXT 문서 합치기
all_docs = pdf_docs + [text_doc]  # ⭐ 리스트로 감싸기!

print(f"전체 문서 개수: {len(all_docs)}")
print(f"  - PDF 문서: {len(pdf_docs)}개")
print(f"  - TXT 문서: 1개")
print(f"\n마지막 문서(TXT) 메타데이터:")
print(all_docs[-1].metadata)

전체 문서 개수: 16
  - PDF 문서: 15개
  - TXT 문서: 1개

마지막 문서(TXT) 메타데이터:
{'source': '파생상품시장 업무규정 시행세칙_전문'}


## 2. 텍스트 분할/문서 청킹 (Text Splitting)

**왜 필요한가?**
- PDF 15페이지 + 긴 TXT 파일 → 한 번에 임베딩하면 너무 큼
- LLM의 컨텍스트 윈도우 제한 (gpt-4o-mini: 128k 토큰)
- 작은 조각으로 나눠야 검색 정확도 ↑

**어떻게?**
- RecursiveCharacterTextSplitter 사용
- 법률 문서 특성: 조문 단위로 자르기 권장

**참고:**
- PRJ01_W2_004_Text_Splitter.ipynb
- chunk_size: 500-1000 (법률 문서는 500 권장)
- chunk_overlap: 100 (문맥 유지)

In [14]:
print(f'다섯 번째 문서: {all_docs[4]}')

다섯 번째 문서: page_content='- 5 -
<신설 2025. 12. 15.>⑤ 회원에게 배정하는 통신회선의 용량은 다음 각 호의 범위 내로 한다. <신설 2012. 4. 27., 2015. 4. 13., 2022. 5. 18.,개정 2025. 12. 15.>1. 증권시장의 경우 45M2. 파생상품시장의 경우 45M⑥ 2개 이상의 회원전산센터를 설치한 회원은 각 회원전산센터에 연결된 각 통신회선의 용량이 동일하도록 관리하여야 한다. <신설 2012. 4. 27.,개정 2025. 5. 30.>⑦ 거래소는 회원에게 배정할 수 있는 증권시장과 파생상품시장별 통신회선의 최대 용량을 정하거나 변경하는 경우 이를 즉시 회원에게 통지한다.제6조(통신회선의 신청) ① 회원은 제5조제2항 각 호에서 정한 통신회선의 범위 내에서 거래소에 통신회선의 신규배정, 추가배정 또는 변경배정을 신청할 수 있다. <개정 2012. 4. 27.,2025. 5. 30.>② 제1항에 따라 파생상품시장의 통신회선을 신청하는 회원은 회원전산센터별로 파생상품거래공유회선 또는 파생상품거래공유회선이 아닌 통신회선 중 하나로만 구성되도록 신청하여야 한다. <신설 2025. 5. 30.>③ 제1항의 신청은 별지 제1호 서식에 따라 통신회선 변경일 전 10거래일까지 거래소에 신청한다.④ 회원의 합병·분할·이전 등으로 인하여 회원시스템이 변경되는 경우의 통신회선의 배정 및 신청은 거래소가 그때마다 정하는 바에 따른다. <개정 2015. 8. 7.>제7조(네트워크의 연결) ① 회원은 거래소시스템과 연결하는 회원시스템의 네트워크를 변경하려는 경우 별지 제2호 서식에 따라 네트' metadata={'producer': 'Call PDF v 2.4', 'creator': 'Call PDF', 'creationdate': '', 'title': '', 'author': '', 'subject': '', 'source': '.././data/membersystem_access_rules.pdf', 'total_p

In [15]:
pdf_docs[4].page_content

'- 5 -\n<신설 2025. 12. 15.>⑤ 회원에게 배정하는 통신회선의 용량은 다음 각 호의 범위 내로 한다. <신설 2012. 4. 27., 2015. 4. 13., 2022. 5. 18.,개정 2025. 12. 15.>1. 증권시장의 경우 45M2. 파생상품시장의 경우 45M⑥ 2개 이상의 회원전산센터를 설치한 회원은 각 회원전산센터에 연결된 각 통신회선의 용량이 동일하도록 관리하여야 한다. <신설 2012. 4. 27.,개정 2025. 5. 30.>⑦ 거래소는 회원에게 배정할 수 있는 증권시장과 파생상품시장별 통신회선의 최대 용량을 정하거나 변경하는 경우 이를 즉시 회원에게 통지한다.제6조(통신회선의 신청) ① 회원은 제5조제2항 각 호에서 정한 통신회선의 범위 내에서 거래소에 통신회선의 신규배정, 추가배정 또는 변경배정을 신청할 수 있다. <개정 2012. 4. 27.,2025. 5. 30.>② 제1항에 따라 파생상품시장의 통신회선을 신청하는 회원은 회원전산센터별로 파생상품거래공유회선 또는 파생상품거래공유회선이 아닌 통신회선 중 하나로만 구성되도록 신청하여야 한다. <신설 2025. 5. 30.>③ 제1항의 신청은 별지 제1호 서식에 따라 통신회선 변경일 전 10거래일까지 거래소에 신청한다.④ 회원의 합병·분할·이전 등으로 인하여 회원시스템이 변경되는 경우의 통신회선의 배정 및 신청은 거래소가 그때마다 정하는 바에 따른다. <개정 2015. 8. 7.>제7조(네트워크의 연결) ① 회원은 거래소시스템과 연결하는 회원시스템의 네트워크를 변경하려는 경우 별지 제2호 서식에 따라 네트'

In [16]:
long_text = pdf_docs[4].page_content
print(f'다섯 번째 문서의 텍스트 길이: {len(long_text)}')

다섯 번째 문서의 텍스트 길이: 788


In [ ]:
# 각 문서의 텍스트 길이 확인

# 방법 1: 기본 반복문
print("=" * 60)
print("📊 문서별 텍스트 길이")
print("=" * 60)

for i, doc in enumerate(all_docs):
    # i: 인덱스 (0부터 시작)
    # doc: Document 객체
    text_length = len(doc.page_content)
    source = doc.metadata.get("source", "알 수 없음")
    
    print(f"{i+1:2d}. 길이: {text_length:6,d}자 | 출처: {source}")

# 방법 2: 요약 정보
print("\n" + "=" * 60)
print("📈 요약 통계")
print("=" * 60)

# 모든 문서의 길이 리스트
lengths = [len(doc.page_content) for doc in all_docs]

print(f"총 문서 수: {len(all_docs)}개")
print(f"전체 텍스트 길이: {sum(lengths):,}자")
print(f"평균 길이: {sum(lengths) / len(lengths):.0f}자")
print(f"최소 길이: {min(lengths):,}자")
print(f"최대 길이: {max(lengths):,}자")

📊 문서별 텍스트 길이
 1. 길이:    921자 | 출처: .././data/membersystem_access_rules.pdf
 2. 길이:    926자 | 출처: .././data/membersystem_access_rules.pdf
 3. 길이:    819자 | 출처: .././data/membersystem_access_rules.pdf
 4. 길이:    799자 | 출처: .././data/membersystem_access_rules.pdf
 5. 길이:    788자 | 출처: .././data/membersystem_access_rules.pdf
 6. 길이:    870자 | 출처: .././data/membersystem_access_rules.pdf
 7. 길이:    847자 | 출처: .././data/membersystem_access_rules.pdf
 8. 길이:    845자 | 출처: .././data/membersystem_access_rules.pdf
 9. 길이:    765자 | 출처: .././data/membersystem_access_rules.pdf
10. 길이:    564자 | 출처: .././data/membersystem_access_rules.pdf
11. 길이:    621자 | 출처: .././data/membersystem_access_rules.pdf
12. 길이:    463자 | 출처: .././data/membersystem_access_rules.pdf
13. 길이:    190자 | 출처: .././data/membersystem_access_rules.pdf
14. 길이:    137자 | 출처: .././data/membersystem_access_rules.pdf
15. 길이:    338자 | 출처: .././data/membersystem_access_rules.pdf
16. 길이: 251,357자 | 출처: 파생상품시장 업무규정 시행세칙_전문

📈 요약 통계
총 문서 

In [ ]:
# TODO: 여기에 청킹 코드를 작성하세요!
# 
# 힌트:
# 1. from langchain_text_splitters import RecursiveCharacterTextSplitter
# 2. text_splitter = RecursiveCharacterTextSplitter(
#        chunk_size=???,
#        chunk_overlap=???,
#        ...
#    )
# 3. chunks = text_splitter.split_documents(all_docs)
# 4. print(f"생성된 청크 수: {len(chunks)}")
#
# 여기서부터 직접 작성해보세요! ↓

- HuggingFace에서 지원하는 BAAI/bge-m3 임베딩 모델을 사용하여 문서를 벡터화할 예정임
- 법률 문서 RAG의 경우 → 같은 토크나이저 사용 권장! ⭐⭐⭐⭐⭐ (Claude ai)
  1. 정확성 중요 - 법률 용어 정확히 처리
  2. 긴 문서 - max_length 초과 방지
  3. 프로덕션급 - 나중에 리팩토링 불필요

In [24]:
# Step 1: 임베딩 모델 먼저 결정
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# Step 2: 같은 모델의 토크나이저 가져오기
tokenizer = embeddings_model.client.tokenizer

# 토크나이저 설정 확인
print(tokenizer.model_max_length)  # 최대 토큰 길이 = 8192
print(tokenizer.vocab_size)        # 어휘 크기 = 250002

# 토큰 수를 계산하는 함수
def count_tokens(text):
    return len(tokenizer(text)['input_ids'])

8192
250002


In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                      
    chunk_overlap=100, # 중첩          
    length_function=count_tokens,         # 토큰 수를 기준으로 분할
    separators=["\n\n", "\n",],   # 구분자 - 재귀적으로 순차적으로 적용 
)

# 텍스트 분할
chunks = text_splitter.split_documents(all_docs)
print(f"생성된 텍스트 청크 수: {len(chunks)}")
print(f"각 청크의 길이: {list(len(chunk.page_content) for chunk in chunks)}")
print(f"각 청크의 토큰 수: {list(count_tokens(chunk.page_content) for chunk in chunks)}")

생성된 텍스트 청크 수: 432
각 청크의 길이: [921, 926, 819, 799, 788, 870, 847, 845, 765, 564, 621, 463, 190, 137, 338, 992, 1004, 985, 987, 912, 622, 344, 868, 364, 940, 835, 806, 934, 234, 760, 780, 778, 793, 392, 852, 838, 768, 526, 596, 745, 720, 733, 750, 709, 624, 345, 815, 707, 615, 810, 721, 731, 725, 554, 752, 377, 575, 667, 716, 409, 736, 667, 789, 777, 820, 424, 848, 844, 659, 428, 547, 864, 493, 121, 784, 621, 276, 278, 707, 709, 520, 732, 719, 734, 798, 822, 532, 725, 737, 804, 370, 829, 735, 103, 818, 389, 609, 866, 477, 741, 615, 889, 890, 847, 109, 762, 229, 293, 743, 741, 604, 925, 520, 840, 368, 583, 739, 780, 340, 463, 671, 541, 695, 669, 701, 742, 501, 719, 986, 404, 540, 556, 771, 735, 564, 602, 774, 364, 792, 615, 610, 470, 548, 705, 867, 831, 147, 837, 317, 524, 816, 754, 543, 847, 832, 601, 770, 819, 761, 733, 758, 283, 761, 655, 620, 326, 724, 428, 718, 814, 653, 251, 793, 768, 825, 191, 757, 512, 657, 556, 773, 210, 512, 845, 333, 486, 769, 634, 774, 610, 328, 841, 216, 523, 

In [26]:
# 청크의 텍스트 확인
print(chunks[20].page_content)

개정 2025. 3. 24. 규정 제2313호
개정 2025. 4. 18. 규정 제2324호
개정 2025. 5. 29. 규정 제2334호
개정 2025. 8. 18. 규정 제2353호
개정 2025. 10. 1. 규정 제2359호
개정 2026. 2. 9. 규정 제2409호
제1편 총칙 
제1조(목적) 이 세칙은 「파생상품시장 업무규정」의 시행에 필요한 사항을 규정함을 목적으로 한다.<개정 2009. 10. 26.>
제2조(정의) ① 이 세칙에서 사용하는 용어의 뜻은 다음과 같다.<개정 2014. 8. 28.>
1. “기초자산기준가격”이란 별표 1에서 정하는 기초자산의 가격(그 가격이 없는 경우에는 순차적으로 앞당긴 날의 가격으로 한다)을 말한다.
2. "영업일"이란 장종료 시점이 속하는 날을 말한다.<2009. 10. 26.,개정 2025. 5. 29.>
3. “의제약정가격”이란 선물스프레드거래의 성립에 따라 거래가 체결된 것으로 보는 선물거래 결제월종목의 약정가격을 말한다.
② 제1항 이외에 이 세칙에서 사용하는 용어의 뜻은 이 세칙에서 특별히 정하는 경우를 제외하고는 법, 그 밖에 거래소 업무관련규정에서 정하는 바에 따른다.<신설 2025. 5. 29.>
[전문개정 2009. 1. 20.]


In [27]:
# 청크의 텍스트 확인
print(chunks[20].metadata)

{'source': '파생상품시장 업무규정 시행세칙_전문'}


## 3. 벡터 저장소 기반 RAG 검색기 (Retriever)


### `3-(1) 벡터 저장소 설정`
- HuggingFace에서 지원하는 BAAI/bge-m3 임베딩 모델을 사용하여 문서를 벡터화
- FAISS DB를 벡터 스토어로 사용 (IndexFlatL2 사용: 유클리드 거리)

In [ ]:
# 임베딩 차원 확인
embedding = embeddings_model.embed_query("test")
print(f"임베딩 차원: {len(embedding)}") # 1024

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


임베딩 차원: 1024


In [31]:
# Ollama 임베딩 모델을 사용한 FAISS 벡터 저장소 생성
import faiss 
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# FAISS 인덱스 초기화 (유클리드 거리 사용)
dim = 1024  # 임베딩 차원
faiss_index = faiss.IndexFlatL2(dim)  
print("=== FAISS 인덱스 초기화 완료")

# FAISS 벡터 저장소의 벡터 차원 수 (임베딩 차원 수)
faiss_index.d

=== FAISS 인덱스 초기화 완료


1024

In [34]:
# FAISS 벡터 저장소 생성
faiss_db = FAISS(
    embedding_function=embeddings_model,
    index=faiss_index,           # 벡터 검색을 위한 데이터 구조를 정의
    docstore=InMemoryDocstore(), # 문서 저장소 객체를 지정 - 문서의 원본 내용과 메타데이터를 메모리에 보관
    index_to_docstore_id={},     # 인덱스와 문서 간의 연결을 관리 (매핑 딕셔너리)
)

# 저장된 문서의 갯수 확인
faiss_db.index.ntotal

0

In [ ]:
import uuid

# 문서 id 생성
doc_ids = [str(uuid.uuid4()) for _ in range(len(chunks))]

# 문서를 벡터 저장소에 저장
# 힌트: faiss_db.add_documents(chunks, ids=doc_ids) 사용
added_doc_ids = faiss_db.add_documents(chunks, ids=doc_ids)

# 벡터 저장소에 저장된 문서를 확인
print(f"=== {len(added_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(added_doc_ids)

In [36]:
# 저장된 문서의 갯수 확인
faiss_db.index.ntotal

0

In [ ]:
# 저장된 인덱스 확인
faiss_db.index_to_docstore_id

In [ ]:
# 저장된 문서 검색
faiss_db.docstore.search('DOC_1')

### `3-(2) 검색기 정의`
- mmr 검색으로 상위 3개 문서 검색하는 Retriever 사용
- 다양성을 높이는 설정을 사용 

In [ ]:
# mmr 검색기 생성
# 힌트: faiss_db.as_retriever(search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 10, 'lambda_mult': 0.3})
# lambda_mult를 낮게 설정하여 다양성을 높임
faiss_mmr_retriever = faiss_db.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': 3,
        'fetch_k': 10,
        'lambda_mult': 0.3
    }
)

In [ ]:
# 검색 테스트 
query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
# 힌트: faiss_mmr_retriever.invoke(query) 사용
retrieved_docs = faiss_mmr_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]}")
    print("-" * 100)

### `3-(3) RAG 프롬프트 구성`

- 작성 기준: 
    - LangChain의 ChatPromptTemplate 클래스 사용
    - 변수 처리는 {context}, {question} 형식 사용
    - 답변은 한글로 출력되도록 프롬프트 작성
    
- 아래 템플릿 코드를 기반으로 다음 내용을 참고하여 작성합니다. 

    1. 프롬프트 구성요소:
        - 작업 지침
        - 컨텍스트 영역
        - 질문 영역
        - 답변 형식 가이드

    2. 작업 지침:
        - 컨텍스트 기반 답변 원칙
        - 외부 지식 사용 제한
        - 불확실성 처리 방법
        - 답변 불가능한 경우의 처리 방법

    3. 답변 형식:
        - 핵심 답변 섹션
        - 근거 제시 섹션
        - 추가 설명 섹션 (필요시)

    4. 제약사항 반영:
        - 답변은 사실에 기반해야 함
        - 추측이나 가정을 최소화해야 함
        - 명확한 근거 제시가 필요함
        - 구조화된 형태로 작성되어야 함

In [ ]:
# Prompt 템플릿 (여기에 작성하세요)
from langchain_core.prompts import ChatPromptTemplate

template = """당신은 KRX(한국거래소) 현행 규정 전문가입니다.

[작업 지침]
- 반드시 아래 [컨텍스트]만을 근거로 답변하세요.
- 외부 지식 사용은 제한하세요.
- 아래 [컨텍스트]에 근거했을 때 불확실한 경우 불확실하다고 답변하세요.
- 답변 불가능한 경우 답변 불가능하다고 답변하세요. 이 때 답변 불가능한 구체적이고 논리적인 이유를 함께 제시하세요.
- 추측이나 가정은 최소화하세요.
- 명확한 근거 제시가 필요합니다.
- 구조화된 형태로 작성되어야 합니다.
- 한국어로 답변하세요.

[컨텍스트]
{context}

[질문] 
{question}

[답변 형식 가이드]
- 핵심 답변 섹션: 주요 내용을 요약하여 핵심만 정리합니다.
- 근거 제시 섹션: 위 핵심 답변 섹션에 언급한 각 항목에 대해 [컨텍스트]에서 찾은 모든 내용을 정확하고 구체적으로 인용하세요. 구조화된 bullet point 형식으로 답변하세요.
- 추가 안내 섹션(필요시): 질문에 완전히 답변하려면 어떤 추가 문서나 규정을 확인해야 하는지 안내해 주세요. 외부 지식을 직접 제공하지 말고, 참조해야 할 문서/규정명만 안내하세요.
"""

prompt = ChatPromptTemplate.from_template(template)

# 템플릿 출력
prompt.pretty_print()

### `3-(4) RAG 체인 구성`
- LangChain의 LCEL 문법을 사용
- 검색 결과를 프롬프트의 'context'로 전달하고,
- 사용자가 입력한 질문을 그대로 프롬프트의 'question'에 전달
- LLM 설정:
    - ChatOpenAI 사용 ('gpt-4o-mini' 모델)
    - temperature: 답변의 일관성을 가져가는 설정값을 사용 
    - 기타 필요한 설정 
- 출력 파서: 문자열 부분만 출력되도록 구성

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# LLM 설정
# 힌트: ChatOpenAI(model='gpt-4o-mini', temperature=0) 사용
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# 문서 포맷팅 (출처 정보 포함 → LLM이 근거 제시 섹션에서 출처 활용 가능)
def format_docs(docs):
    return "\n\n".join([
        f"[출처: {doc.metadata.get('source', '불명')}]\n{doc.page_content}"
        for doc in docs
    ])

# RAG 체인 생성
# 힌트: {'context': faiss_mmr_retriever | format_docs, 'question': RunnablePassthrough()} | prompt | llm | StrOutputParser()
rag_chain = (
    {
        'context': faiss_mmr_retriever | format_docs, 
        'question': RunnablePassthrough()
    }
    | prompt 
    | llm 
    | StrOutputParser()
)

# 체인 실행
query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
output = rag_chain.invoke(query)

print(f"쿼리: {query}")
print("답변:")
print(output)

### `3-(5) Gradio 스트리밍 구현`
- ChatInterface 사용
- `chain.stream()`으로 응답을 청크 단위로 스트리밍

In [ ]:
import gradio as gr
from typing import Iterator

# 스트리밍 응답 생성 함수
def get_streaming_response(message: str, history) -> Iterator[str]:
    
    # RAG Chain 실행 및 스트리밍 응답 생성
    response = ""
    for chunk in rag_chain.stream(message):
        if isinstance(chunk, str):
            response += chunk
            yield response

# Gradio 인터페이스 설정
# 힌트: gr.ChatInterface(fn=get_streaming_response, title="RAG 기반 질의응답 시스템", description="...", examples=[...])
demo = None

# 실행
demo.launch()

In [ ]:
# demo 실행 종료
demo.close()